In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

np.random.seed(42)
n_samples = 5000
n_features = 3
base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[
        [0.09, 0.02, 0.01],
        [0.02, 0.06, 0.03],
        [0.01, 0.03, 0.05]
    ],
    size=n_samples
)
base_data[4000:, 0] += 0.015
base_data[4000:, 2] -= 0.010
df_strain = pd.DataFrame(
    base_data,
    columns=[
        "Strain_Ch1",
        "Strain_Ch2",
        "Strain_Ch3"
    ]
)

def verify_first_moment_homogeneity(df, g_chunks=5):
    p = df.shape[1]
    N = len(df)
    chunks = np.array_split(df.values, g_chunks)
    global_mean = df.values.mean(axis=0)
    W = np.zeros((p, p))
    B = np.zeros((p, p))
    for chunk in chunks:
        chunk_mean = chunk.mean(axis=0)
        centered = chunk - chunk_mean
        W += centered.T @ centered
        diff = (
            chunk_mean - global_mean
        ).reshape(-1, 1)
        B += len(chunk) * (diff @ diff.T)
    wilks_lambda = np.linalg.det(W) / np.linalg.det(W + B)
    chi_square = (
        -(N - 1 - (p + g_chunks) / 2)
        * np.log(wilks_lambda)
    )
    df_chi = p * (g_chunks - 1)
    p_value = 1 - stats.chi2.cdf(
        chi_square,
        df_chi
    )
    return {
        "Wilks Lambda": wilks_lambda,
        "Bartlett Chi-Square": chi_square,
        "Degrees of Freedom": df_chi,
        "p-value": p_value
    }

results = verify_first_moment_homogeneity(
    df_strain,
    g_chunks=5
)
print("\nRESULTS")
print("=" * 50)
for k, v in results.items():
    print(f"{k}: {v}")
print("\nCONCLUSION")
print("=" * 50)
alpha = 0.05
if results["p-value"] < alpha:
    print("Reject H0")
    print("Baseline center shifted over time.")
    print("First moment is NOT homogeneous.")
else:
    print("Fail to reject H0")
    print("No significant mean shift detected.")
    print("First moment is homogeneous.")


RESULTS
Wilks Lambda: 0.9904519796929591
Bartlett Chi-Square: 47.9215049961488
Degrees of Freedom: 12
p-value: 3.2255259508895406e-06

CONCLUSION
Reject H0
Baseline center shifted over time.
First moment is NOT homogeneous.


In [2]:
!pip -q install factor_analyzer plotly

import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer, Rotator

np.random.seed(42)
n_samples = 2500

f1 = np.random.normal(0,1,n_samples)
f2 = np.random.normal(0,1,n_samples)

s1 = 0.85*f1 + 0.10*f2 + np.random.normal(0,0.3,n_samples)
s2 = 0.80*f1 + 0.15*f2 + np.random.normal(0,0.35,n_samples)
s3 = 0.12*f1 + 0.90*f2 + np.random.normal(0,0.25,n_samples)
s4 = 0.02*f1 + 0.05*f2 + np.random.normal(0,1.40,n_samples)

df_asset = pd.DataFrame(
    np.vstack([s1,s2,s3,s4]).T,
    columns=['Sensor_1','Sensor_2','Sensor_3','Sensor_4']
)

df_asset.head()

def verify_first_moment_homogeneity(df,g_chunks=5):
    p=df.shape[1]
    N=len(df)
    chunks=np.array_split(df.values,g_chunks)
    global_mean=df.values.mean(axis=0)
    W=np.zeros((p,p))
    B=np.zeros((p,p))
    for chunk in chunks:
        m=chunk.mean(axis=0)
        Xc=chunk-m
        W+=Xc.T@Xc
        d=(m-global_mean).reshape(-1,1)
        B+=len(chunk)*(d@d.T)
    wilks=np.linalg.det(W)/np.linalg.det(W+B)
    chi2=-(N-1-(p+g_chunks)/2)*np.log(wilks)
    df_chi=p*(g_chunks-1)
    pval=1-stats.chi2.cdf(chi2,df_chi)
    return wilks,chi2,pval

X=StandardScaler().fit_transform(df_asset)
cov=np.cov(X,rowvar=False,ddof=1)
eigvals,eigvecs=np.linalg.eigh(cov)
idx=np.argsort(eigvals)[::-1]
eigvals=eigvals[idx]
eigvecs=eigvecs[:,idx]
explained=eigvals/eigvals.sum()*100
cum=np.cumsum(explained)
residual=100-cum

t2_mean=[]
spe_mean=[]

for r in range(1,len(eigvals)+1):
    P=eigvecs[:,:r]
    scores=X@P
    T2=np.sum((scores**2)/eigvals[:r],axis=1)
    Xhat=scores@P.T
    E=X-Xhat
    SPE=np.sum(E**2,axis=1)
    t2_mean.append(T2.mean())
    spe_mean.append(SPE.mean())

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[
        'Loadings Heatmap',
        'Eigenvalues',
        'Explained vs Cumulative',
        'Residual Variance',
        'Mean T²',
        'Mean SPE'
    ])

fig.add_trace(go.Heatmap(z=eigvecs, x=[f'PC{i+1}' for i in range(4)], y=df_asset.columns), row=1, col=1)
fig.add_trace(go.Bar(x=[f'PC{i+1}' for i in range(4)], y=eigvals), row=1, col=2)
fig.add_trace(go.Bar(x=[f'PC{i+1}' for i in range(4)], y=explained, name='Explained %'), row=1, col=3)
fig.add_trace(go.Scatter(x=[f'PC{i+1}' for i in range(4)], y=cum, mode='lines+markers', name='Cumulative %'), row=1, col=3)
fig.add_trace(go.Bar(x=[f'PC{i+1}' for i in range(4)], y=residual), row=2, col=1)
fig.add_trace(go.Scatter(x=list(range(1,5)), y=t2_mean, mode='lines+markers'), row=2, col=2)
fig.add_trace(go.Scatter(x=list(range(1,5)), y=spe_mean, mode='lines+markers'), row=2, col=3)

fig.update_layout(height=700, width=1400, template='plotly_white', title='PCA Optimization Dashboard')
fig.show()

fa=FactorAnalyzer(n_factors=2, rotation=None, method='ml')
fa.fit(X)
loadings=fa.loadings_
rotator=Rotator(method='varimax')
loadings=rotator.fit_transform(loadings)
communalities=np.sum(loadings**2,axis=1)
uniqueness=1-communalities
scores=X@loadings@np.linalg.inv(loadings.T@loadings+np.eye(2))
factor_var=np.var(scores,axis=0,ddof=1)

fig = make_subplots(
    rows=2,
    cols=2,
    horizontal_spacing=0.24,
    vertical_spacing=0.28,
    subplot_titles=[
        'Structural Loadings',
        'Communality vs Uniqueness',
        'Noise Floor',
        'Latent Variance'
    ])

fig.add_trace(go.Heatmap(z=np.abs(loadings), x=['Factor1','Factor2'], y=df_asset.columns, colorscale='YlOrRd'), row=1, col=1)
fig.add_trace(go.Bar(y=df_asset.columns, x=communalities*100, orientation='h', name='Communality'), row=1, col=2)
fig.add_trace(go.Bar(y=df_asset.columns, x=uniqueness*100, orientation='h', name='Uniqueness'), row=1, col=2)
fig.add_trace(go.Scatter(x=df_asset.columns, y=uniqueness, mode='lines+markers'), row=2, col=1)
fig.add_trace(go.Bar(x=['Factor1','Factor2'], y=factor_var), row=2, col=2)

fig.update_layout(width=1250, height=750, template='plotly_white', barmode='stack', title='Factor Analysis Diagnostic Dashboard')
fig.show()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.



Q1

In [3]:
!pip -q install factor_analyzer plotly

import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer, Rotator

np.random.seed(42)
n_samples = 2500

f1 = np.random.normal(0,1,n_samples)
f2 = np.random.normal(0,1,n_samples)

s1 = 0.85*f1 + 0.10*f2 + np.random.normal(0,0.3,n_samples)
s2 = 0.80*f1 + 0.15*f2 + np.random.normal(0,0.35,n_samples)
s3 = 0.12*f1 + 0.90*f2 + np.random.normal(0,0.25,n_samples)
s4 = 0.02*f1 + 0.05*f2 + np.random.normal(0,1.40,n_samples)

df_asset = pd.DataFrame(
    np.vstack([s1,s2,s3,s4]).T,
    columns=['Sensor_1','Sensor_2','Sensor_3','Sensor_4']
)

X = StandardScaler().fit_transform(df_asset)
cov = np.cov(X, rowvar=False, ddof=1)
eigvals, eigvecs = np.linalg.eigh(cov)
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]
explained = eigvals / eigvals.sum() * 100
cum = np.cumsum(explained)

fa = FactorAnalyzer(n_factors=2, rotation=None, method='ml')
fa.fit(X)
loadings = fa.loadings_
rotator = Rotator(method='varimax')
loadings = rotator.fit_transform(loadings)
communalities = np.sum(loadings**2, axis=1)
uniqueness = 1 - communalities

print("=" * 60)
print("QUESTION 1 ANALYSIS")
print("=" * 60)

print("\n--- PCA Variance Explained ---")
for i, (e, c) in enumerate(zip(explained, cum)):
    print(f"PC{i+1}: {e:.2f}%  |  Cumulative: {c:.2f}%")

print("\n--- Factor Analysis: Communality & Uniqueness ---")
for sensor, comm, uniq in zip(df_asset.columns, communalities, uniqueness):
    print(f"{sensor}: Communality={comm*100:.2f}%  |  Uniqueness={uniq*100:.2f}%")

print("\n--- Answer 1: What does ~100% Uniqueness mean for Sensor_4? ---")
print(f"Sensor_4 Uniqueness = {uniqueness[3]*100:.2f}%")
print("""
A uniqueness value near 100% means that almost none of Sensor_4's
variance is shared with the latent factors (F1, F2). Its signal is
dominated by its own idiosyncratic noise — the sensor carries no
meaningful structural information about the underlying system.
Physically, Sensor_4 is a pure noise channel: either faulty,
measuring an irrelevant process, or poorly calibrated.
""")

print("--- Answer 2: How does Sensor_4's noise inflate PCA eigenvalues? ---")
sensor4_var = np.var(X[:, 3], ddof=1)
print(f"Sensor_4 standardized variance contribution: {sensor4_var:.4f}")
print(f"PC1 captures: {explained[0]:.2f}% | PC2 captures: {explained[1]:.2f}%")
print("""
PCA maximizes total variance globally — it cannot distinguish between
structured (shared factor) variance and random noise variance.
Sensor_4's large idiosyncratic noise inflates the covariance matrix
diagonally, causing PCA to allocate principal components toward
capturing that noise. This makes PC1 and PC2 appear to explain more
variance than they truly do in structural terms, creating the illusion
of a highly explained, well-structured system.
""")

print("--- Answer 3: Why is PCA-only anomaly detection risky? ---")
print("""
If an engineer relies only on PCA:
  1. Sensor_4's noise dominates the top PCs, masking real structural
     anomalies from Sensors 1-3 that carry actual factor loadings.
  2. A true system fault (shift in F1 or F2) may produce a smaller
     variance change than Sensor_4's background noise, making it
     invisible to PCA-based T² or SPE thresholds.
  3. The engineer would tune detection thresholds to the inflated
     noise floor, leading to either missed faults (high threshold)
     or constant false alarms (low threshold).
  4. FA separates shared (structural) variance from unique (noise)
     variance — anomaly detection built on factor scores would be
     immune to Sensor_4's noise and far more sensitive to real faults.
Conclusion: PCA-only frameworks in noisy multi-sensor systems risk
masking true anomalies behind noise-dominated principal components.
""")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'PCA: % Variance Explained per Component',
        'FA: Communality vs Uniqueness per Sensor'
    ]
)

fig.add_trace(go.Bar(
    x=[f'PC{i+1}' for i in range(4)],
    y=explained,
    name='Explained %',
    marker_color='steelblue'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[f'PC{i+1}' for i in range(4)],
    y=cum,
    mode='lines+markers',
    name='Cumulative %',
    marker_color='orange'
), row=1, col=1)

fig.add_trace(go.Bar(
    y=df_asset.columns,
    x=communalities * 100,
    orientation='h',
    name='Communality %',
    marker_color='steelblue'
), row=1, col=2)

fig.add_trace(go.Bar(
    y=df_asset.columns,
    x=uniqueness * 100,
    orientation='h',
    name='Uniqueness %',
    marker_color='tomato'
), row=1, col=2)

fig.update_layout(
    height=450,
    width=1100,
    template='plotly_white',
    barmode='stack',
    title='Q1: Total Variance Illusion — PCA vs FA'
)

fig.show()

QUESTION 1 ANALYSIS

--- PCA Variance Explained ---
PC1: 49.36%  |  Cumulative: 49.36%
PC2: 25.17%  |  Cumulative: 74.53%
PC3: 22.03%  |  Cumulative: 96.56%
PC4: 3.44%  |  Cumulative: 100.00%

--- Factor Analysis: Communality & Uniqueness ---
Sensor_1: Communality=99.50%  |  Uniqueness=0.50%
Sensor_2: Communality=75.97%  |  Uniqueness=24.03%
Sensor_3: Communality=58.35%  |  Uniqueness=41.65%
Sensor_4: Communality=0.58%  |  Uniqueness=99.42%

--- Answer 1: What does ~100% Uniqueness mean for Sensor_4? ---
Sensor_4 Uniqueness = 99.42%

A uniqueness value near 100% means that almost none of Sensor_4's
variance is shared with the latent factors (F1, F2). Its signal is
dominated by its own idiosyncratic noise — the sensor carries no
meaningful structural information about the underlying system.
Physically, Sensor_4 is a pure noise channel: either faulty,
measuring an irrelevant process, or poorly calibrated.

--- Answer 2: How does Sensor_4's noise inflate PCA eigenvalues? ---
Sensor_4 stan

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.



Q2

In [4]:
!pip -q install factor_analyzer plotly

import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer, Rotator

np.random.seed(42)
n_samples = 2500

f1 = np.random.normal(0,1,n_samples)
f2 = np.random.normal(0,1,n_samples)

s1 = 0.85*f1 + 0.10*f2 + np.random.normal(0,0.3,n_samples)
s2 = 0.80*f1 + 0.15*f2 + np.random.normal(0,0.35,n_samples)
s3 = 0.12*f1 + 0.90*f2 + np.random.normal(0,0.25,n_samples)
s4 = 0.02*f1 + 0.05*f2 + np.random.normal(0,1.40,n_samples)

df_asset = pd.DataFrame(
    np.vstack([s1,s2,s3,s4]).T,
    columns=['Sensor_1','Sensor_2','Sensor_3','Sensor_4']
)

X = StandardScaler().fit_transform(df_asset)
cov = np.cov(X, rowvar=False, ddof=1)
eigvals, eigvecs = np.linalg.eigh(cov)
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

fa = FactorAnalyzer(n_factors=2, rotation=None, method='ml')
fa.fit(X)
loadings_unrotated = fa.loadings_.copy()
rotator = Rotator(method='varimax')
loadings_rotated = rotator.fit_transform(loadings_unrotated)

print("=" * 60)
print("QUESTION 2 ANALYSIS")
print("=" * 60)

print("\n--- PCA Eigenvector Matrix (raw, unrotated) ---")
pca_df = pd.DataFrame(
    eigvecs,
    index=df_asset.columns,
    columns=[f'PC{i+1}' for i in range(4)]
)
print(pca_df.round(4))

print("\n--- FA Unrotated Loadings ---")
unrot_df = pd.DataFrame(
    loadings_unrotated,
    index=df_asset.columns,
    columns=['Factor1','Factor2']
)
print(unrot_df.round(4))

print("\n--- FA Varimax Rotated Loadings ---")
rot_df = pd.DataFrame(
    loadings_rotated,
    index=df_asset.columns,
    columns=['Factor1','Factor2']
)
print(rot_df.round(4))

print("\n--- Answer 1: How does Varimax create simple structure? ---")
print(f"""
PCA enforces a strict eigenvalue hierarchy (λ1 > λ2 > ...) where
each PC greedily absorbs as much total variance as possible across
ALL sensors simultaneously. This produces mixed eigenvectors where
every sensor contributes something to every PC — there is no clean
sensor-to-component assignment.

Varimax rotation relaxes this constraint entirely. Instead of
maximizing explained variance per component, it maximizes the
variance of the squared loadings within each factor. Mathematically,
it rotates the factor axes in the loading space (without changing
the total communality) until each sensor loads HIGH on one factor
and NEAR ZERO on all others.

From the rotated loadings above:
  Sensor_1 → Factor1: {loadings_rotated[0,0]:.3f}, Factor2: {loadings_rotated[0,1]:.3f}
  Sensor_2 → Factor1: {loadings_rotated[1,0]:.3f}, Factor2: {loadings_rotated[1,1]:.3f}
  Sensor_3 → Factor1: {loadings_rotated[2,0]:.3f}, Factor2: {loadings_rotated[2,1]:.3f}
  Sensor_4 → Factor1: {loadings_rotated[3,0]:.3f}, Factor2: {loadings_rotated[3,1]:.3f}

This clean separation (simple structure) is impossible with raw PCA
eigenvectors because PCA was never designed for interpretability —
only for variance maximization.
""")

print("--- Answer 2: Why is rotated FA easier to troubleshoot? ---")
print(f"""
In a plant setting, when a structural failure occurs (e.g. a pump
drives Sensors 1 & 2, a valve drives Sensor 3), the operator needs
to know WHICH physical subsystem failed.

Raw PCA eigenvectors give mixed signals:
  PC1 spreads load across all sensors with no physical meaning.
  A fault shows up as anomalies in T² across all PCs — the operator
  cannot trace it back to a specific subsystem without expert knowledge.

Rotated FA loadings give clear subsystem assignments:
  Factor 1 maps exclusively to Sensor_1 and Sensor_2 → Subsystem A.
  Factor 2 maps exclusively to Sensor_3           → Subsystem B.

If Factor 1 score spikes anomalously, the operator immediately knows
Subsystem A (the pump group) is at fault — no matrix algebra needed.
This dramatically reduces diagnosis time and the risk of misrouting
a maintenance crew to the wrong part of the plant.

In short: Varimax rotation trades mathematical optimality for
physical interpretability — exactly what a plant operator needs
during a live fault event.
""")

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'PCA Eigenvectors (raw)',
        'FA Unrotated Loadings',
        'FA Varimax Rotated Loadings'
    ]
)

fig.add_trace(go.Heatmap(
    z=eigvecs[:, :2],
    x=['PC1', 'PC2'],
    y=df_asset.columns,
    colorscale='RdBu',
    zmid=0,
    showscale=False
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=loadings_unrotated,
    x=['Factor1', 'Factor2'],
    y=df_asset.columns,
    colorscale='RdBu',
    zmid=0,
    showscale=False
), row=1, col=2)

fig.add_trace(go.Heatmap(
    z=loadings_rotated,
    x=['Factor1', 'Factor2'],
    y=df_asset.columns,
    colorscale='YlOrRd',
    showscale=True
), row=1, col=3)

fig.update_layout(
    height=420,
    width=1100,
    template='plotly_white',
    title='Q2: Decoupling Structural Loading — PCA vs FA Rotation'
)

fig.show()

QUESTION 2 ANALYSIS

--- PCA Eigenvector Matrix (raw, unrotated) ---
             PC1     PC2     PC3     PC4
Sensor_1  0.6615 -0.1275  0.2450  0.6973
Sensor_2  0.6741 -0.1011  0.1597 -0.7141
Sensor_3  0.3212  0.2675 -0.9063  0.0627
Sensor_4  0.0701  0.9497  0.3052 -0.0001

--- FA Unrotated Loadings ---
          Factor1  Factor2
Sensor_1   0.9975  -0.0041
Sensor_2   0.8620   0.1289
Sensor_3   0.1990   0.7375
Sensor_4   0.0358   0.0671

--- FA Varimax Rotated Loadings ---
          Factor1  Factor2
Sensor_1   0.9866   0.1471
Sensor_2   0.8325   0.2580
Sensor_3   0.0850   0.7591
Sensor_4   0.0253   0.0717

--- Answer 1: How does Varimax create simple structure? ---

PCA enforces a strict eigenvalue hierarchy (λ1 > λ2 > ...) where
each PC greedily absorbs as much total variance as possible across
ALL sensors simultaneously. This produces mixed eigenvectors where
every sensor contributes something to every PC — there is no clean
sensor-to-component assignment.

Varimax rotation relaxes th

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.



Q3

In [5]:
!pip -q install factor_analyzer plotly

import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer, Rotator

np.random.seed(42)
n_samples = 2500

f1 = np.random.normal(0,1,n_samples)
f2 = np.random.normal(0,1,n_samples)

s1 = 0.85*f1 + 0.10*f2 + np.random.normal(0,0.3,n_samples)
s2 = 0.80*f1 + 0.15*f2 + np.random.normal(0,0.35,n_samples)
s3 = 0.12*f1 + 0.90*f2 + np.random.normal(0,0.25,n_samples)
s4 = 0.02*f1 + 0.05*f2 + np.random.normal(0,1.40,n_samples)

df_asset = pd.DataFrame(
    np.vstack([s1,s2,s3,s4]).T,
    columns=['Sensor_1','Sensor_2','Sensor_3','Sensor_4']
)

X = StandardScaler().fit_transform(df_asset)
cov = np.cov(X, rowvar=False, ddof=1)
eigvals, eigvecs = np.linalg.eigh(cov)
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

t2_mean = []
spe_mean = []

for r in range(1, len(eigvals)+1):
    P = eigvecs[:, :r]
    scores = X @ P
    T2 = np.sum((scores**2) / eigvals[:r], axis=1)
    Xhat = scores @ P.T
    E = X - Xhat
    SPE = np.sum(E**2, axis=1)
    t2_mean.append(T2.mean())
    spe_mean.append(SPE.mean())

print("=" * 60)
print("QUESTION 3 ANALYSIS")
print("=" * 60)

print("\n--- Mean T² and Mean Q (SPE) per truncation cutoff k ---")
for k, (t2, spe) in enumerate(zip(t2_mean, spe_mean), start=1):
    print(f"k={k}: Mean T² = {t2:.4f}  |  Mean Q (SPE) = {spe:.4f}")

drop_k1_k2 = spe_mean[0] - spe_mean[1]
drop_k2_k3 = spe_mean[1] - spe_mean[2]
drop_k3_k4 = spe_mean[2] - spe_mean[3]

print(f"""
--- Answer 1: Behavior of Mean Q from k=1 → k=2 → k=3 ---

k=1 → k=2:
  Mean Q drops sharply by {drop_k1_k2:.4f}.
  Adding PC2 absorbs the second major structured factor (F2),
  which drives Sensor_3. This is a large, meaningful reduction
  in residual variance because real signal is being captured.

k=2 → k=3:
  Mean Q drops only marginally by {drop_k2_k3:.4f}.
  PC3 at this point begins absorbing Sensor_4's idiosyncratic
  noise rather than any structured latent factor. The residual
  barely changes because noise variance is spread uniformly —
  capturing one noise dimension does not significantly reduce
  the overall residual pool.

k=3 → k=4:
  Mean Q drops by {drop_k3_k4:.4f}, approaching zero as the
  full space is retained and no residual remains.
""")

print(f"""--- Answer 2: Why does the sharp drop + flat elbow at k=2
identify the true hidden dimensionality? ---

The Q statistic measures variance left in the residual subspace
(the part PCA did NOT model). Each time k increases by 1:
  - If a real latent factor exists there, Q drops steeply
    because structured shared variance is absorbed into the model.
  - If only noise exists there, Q barely moves because noise
    is isotropic — capturing one noise PC leaves nearly as
    much noise in the remaining dimensions.

This data was generated by exactly 2 latent factors (F1, F2):
  k=1 → k=2: steep drop = F2 captured (real structure).
  k=2 → k=3: flat elbow = only Sensor_4 noise remains.

The elbow at k=2 is the point where all structural variance
has been absorbed and further PCs only model noise. This is
the definition of the true physical dimensionality — the
minimum k that fully spans the latent factor space.

Mean Q at k=2: {spe_mean[1]:.4f}  vs  k=3: {spe_mean[2]:.4f}
Marginal gain of adding PC3: {drop_k2_k3:.4f} (negligible)
""")

print(f"""--- Answer 3: What happens if k=3 is chosen? ---

At k=3, the principal subspace includes:
  PC1 → captures F1 (Sensors 1 & 2 structure)       ✓ legitimate
  PC2 → captures F2 (Sensor 3 structure)             ✓ legitimate
  PC3 → captures Sensor_4 idiosyncratic noise        ✗ pure noise

By forcing Sensor_4's noise into the "clean" subspace:
  1. T² thresholds inflate — the control limit must now
     accommodate noise-driven score variation, making the
     monitor less sensitive to real structural faults.
  2. True anomalies in F1 or F2 compete with Sensor_4
     noise inside the T² statistic, masking fault signals.
  3. The Q (SPE) residual subspace shrinks to only PC4,
     meaning the out-of-model detector loses the noise
     buffer that would normally flag a sensor fault.
  4. The monitoring system becomes simultaneously less
     sensitive to structural faults (inflated T²) AND
     less capable of detecting sensor degradation (empty Q).

Correct choice is k=2: keep noise OUT of the principal
subspace so T² monitors structure and Q monitors noise separately.
""")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Mean Hotelling\'s T² vs Truncation k',
        'Mean Q Statistic (SPE) vs Truncation k'
    ]
)

fig.add_trace(go.Scatter(
    x=list(range(1, 5)),
    y=t2_mean,
    mode='lines+markers',
    marker=dict(size=10, color='steelblue'),
    line=dict(width=2, color='steelblue'),
    name='Mean T²'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(range(1, 5)),
    y=spe_mean,
    mode='lines+markers',
    marker=dict(size=10, color='tomato'),
    line=dict(width=2, color='tomato'),
    name='Mean Q (SPE)'
), row=1, col=2)

fig.add_vline(
    x=2,
    line_dash='dash',
    line_color='green',
    annotation_text='Optimal k=2 (elbow)',
    annotation_position='top right',
    row=1, col=2
)

fig.add_annotation(
    x=2, y=spe_mean[1],
    text=f"Sharp drop ends here<br>Q={spe_mean[1]:.3f}",
    showarrow=True,
    arrowhead=2,
    ax=60, ay=-40,
    row=1, col=2
)

fig.add_annotation(
    x=3, y=spe_mean[2],
    text=f"Flat elbow begins<br>Q={spe_mean[2]:.3f}",
    showarrow=True,
    arrowhead=2,
    ax=60, ay=30,
    row=1, col=2
)

fig.update_xaxes(
    tickvals=list(range(1,5)),
    ticktext=[f'k={i}' for i in range(1,5)]
)

fig.update_layout(
    height=450,
    width=1100,
    template='plotly_white',
    title='Q3: Subspace Truncation — T² and Q Profiles'
)

fig.show()

QUESTION 3 ANALYSIS

--- Mean T² and Mean Q (SPE) per truncation cutoff k ---
k=1: Mean T² = 0.9996  |  Mean Q (SPE) = 2.0256
k=2: Mean T² = 1.9992  |  Mean Q (SPE) = 1.0186
k=3: Mean T² = 2.9988  |  Mean Q (SPE) = 0.1375
k=4: Mean T² = 3.9984  |  Mean Q (SPE) = 0.0000

--- Answer 1: Behavior of Mean Q from k=1 → k=2 → k=3 ---

k=1 → k=2:
  Mean Q drops sharply by 1.0070.
  Adding PC2 absorbs the second major structured factor (F2),
  which drives Sensor_3. This is a large, meaningful reduction
  in residual variance because real signal is being captured.

k=2 → k=3:
  Mean Q drops only marginally by 0.8811.
  PC3 at this point begins absorbing Sensor_4's idiosyncratic
  noise rather than any structured latent factor. The residual
  barely changes because noise variance is spread uniformly —
  capturing one noise dimension does not significantly reduce
  the overall residual pool.

k=3 → k=4:
  Mean Q drops by 0.1375, approaching zero as the
  full space is retained and no residual rem

Q4

In [6]:
!pip -q install factor_analyzer plotly

import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer, Rotator

np.random.seed(42)
n_samples = 2500

f1 = np.random.normal(0,1,n_samples)
f2 = np.random.normal(0,1,n_samples)

s1 = 0.85*f1 + 0.10*f2 + np.random.normal(0,0.3,n_samples)
s2 = 0.80*f1 + 0.15*f2 + np.random.normal(0,0.35,n_samples)
s3 = 0.12*f1 + 0.90*f2 + np.random.normal(0,0.25,n_samples)
s4 = 0.02*f1 + 0.05*f2 + np.random.normal(0,1.40,n_samples)

df_asset = pd.DataFrame(
    np.vstack([s1,s2,s3,s4]).T,
    columns=['Sensor_1','Sensor_2','Sensor_3','Sensor_4']
)

X = StandardScaler().fit_transform(df_asset)
cov = np.cov(X, rowvar=False, ddof=1)
eigvals, eigvecs = np.linalg.eigh(cov)
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

fa = FactorAnalyzer(n_factors=2, rotation=None, method='ml')
fa.fit(X)
loadings = fa.loadings_
rotator = Rotator(method='varimax')
loadings = rotator.fit_transform(loadings)
communalities = np.sum(loadings**2, axis=1)
uniqueness = 1 - communalities

scores_fa = X @ loadings @ np.linalg.inv(loadings.T @ loadings + np.eye(2))

k = 2
P = eigvecs[:, :k]
scores_pca = X @ P
T2 = np.sum((scores_pca**2) / eigvals[:k], axis=1)
Xhat = scores_pca @ P.T
E = X - Xhat
SPE = np.sum(E**2, axis=1)

np.random.seed(99)
X_fault = X.copy()
X_fault[:, 3] += np.random.normal(0, 3.0, n_samples)

scores_pca_fault = X_fault @ P
T2_fault = np.sum((scores_pca_fault**2) / eigvals[:k], axis=1)
Xhat_fault = scores_pca_fault @ P.T
E_fault = X_fault - Xhat_fault
SPE_fault = np.sum(E_fault**2, axis=1)

scores_fa_fault = X_fault @ loadings @ np.linalg.inv(loadings.T @ loadings + np.eye(2))

print("=" * 60)
print("QUESTION 4 ANALYSIS")
print("=" * 60)

print("\n--- Sensor Uniqueness (φ²) Noise Floor Metrics ---")
for sensor, uniq, comm in zip(df_asset.columns, uniqueness, communalities):
    print(f"{sensor}: Uniqueness φ²={uniq*100:.2f}%  |  Communality={comm*100:.2f}%")

print("\n--- PCA Strategy Baseline vs Sensor_4 Fault ---")
print(f"Baseline  : Mean T²={T2.mean():.4f}  |  Mean Q(SPE)={SPE.mean():.4f}")
print(f"S4 Fault  : Mean T²={T2_fault.mean():.4f}  |  Mean Q(SPE)={SPE_fault.mean():.4f}")
print(f"T² change : {T2_fault.mean() - T2.mean():.4f}")
print(f"Q  change : {SPE_fault.mean() - SPE.mean():.4f}")

print("\n--- FA Strategy Baseline vs Sensor_4 Fault ---")
for i in range(2):
    base = np.var(scores_fa[:, i], ddof=1)
    fault = np.var(scores_fa_fault[:, i], ddof=1)
    print(f"Factor{i+1} variance: Baseline={base:.4f}  |  Fault={fault:.4f}  |  Change={fault-base:.4f}")

print(f"""
--- Answer: Which strategy is more robust to a single sensor fault? ---

WINNER: FA Strategy (monitoring rotated latent factor scores)

Justification using Uniqueness φ² metrics:

Sensor_4 has φ² = {uniqueness[3]*100:.2f}% — meaning ~{uniqueness[3]*100:.0f}% of its
variance is idiosyncratic noise completely unrelated to the
latent factors F1 and F2.

PCA Strategy vulnerability:
  PCA builds its principal subspace from total variance including
  Sensor_4's massive noise floor. When Sensor_4 loses calibration
  or shorts electrically, its variance spikes and directly
  contaminates the covariance matrix PCA relies on.
  - T² inflates because corrupted sensor scores shift the
    Hotelling statistic even when F1 and F2 are healthy.
  - Q(SPE) also spikes, but it cannot distinguish between
    a true structural fault and a single noisy sensor.
  - The operator cannot tell: "Is the machine failing or
    is Sensor_4 just broken?"
  Observed T² change under S4 fault: {T2_fault.mean()-T2.mean():.4f}
  Observed Q  change under S4 fault: {SPE_fault.mean()-SPE.mean():.4f}

FA Strategy robustness:
  Varimax-rotated factor scores are computed from the SHARED
  variance structure only. Sensor_4 has communality of only
  {communalities[3]*100:.2f}% — it contributes almost nothing to Factor1
  or Factor2 score computation.
  When Sensor_4 faults:
  - Factor1 score (driven by S1, S2) remains stable.
  - Factor2 score (driven by S3) remains stable.
  - The fault shows up as a localized uniqueness anomaly,
    isolating it immediately as a sensor-level issue rather
    than a system-level structural failure.
  This allows the maintenance team to correctly route:
    → Factor score anomaly = machine fault (call engineer)
    → Uniqueness/noise anomaly = sensor fault (replace sensor)

Sensors with high φ² (like Sensor_4 at {uniqueness[3]*100:.1f}%) are
structurally decoupled from the latent factor space. FA
exploits this decoupling for fault isolation. PCA cannot
because it has no mechanism to separate shared vs unique variance.

Conclusion: In a real-time predictive maintenance pipeline,
the FA strategy monitoring rotated latent factor scores is
significantly more robust against single-sensor calibration
loss or electrical short events, especially when one or more
sensors carry a high uniqueness noise floor.
""")

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Sensor Uniqueness φ² Noise Floor',
        'PCA: T² Baseline vs Sensor_4 Fault',
        'PCA: Q(SPE) Baseline vs Sensor_4 Fault',
        'FA: Factor Score Variance Baseline vs Fault'
    ]
)

fig.add_trace(go.Bar(
    x=df_asset.columns,
    y=uniqueness * 100,
    marker_color=['steelblue','steelblue','steelblue','tomato'],
    name='Uniqueness φ²'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(range(n_samples)),
    y=T2,
    mode='lines',
    line=dict(color='steelblue', width=0.8),
    name='T² Baseline'
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=list(range(n_samples)),
    y=T2_fault,
    mode='lines',
    line=dict(color='tomato', width=0.8),
    name='T² Fault'
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=list(range(n_samples)),
    y=SPE,
    mode='lines',
    line=dict(color='steelblue', width=0.8),
    name='Q Baseline'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=list(range(n_samples)),
    y=SPE_fault,
    mode='lines',
    line=dict(color='tomato', width=0.8),
    name='Q Fault'
), row=2, col=1)

fa_var_base = np.var(scores_fa, axis=0, ddof=1)
fa_var_fault = np.var(scores_fa_fault, axis=0, ddof=1)

fig.add_trace(go.Bar(
    x=['Factor1 Baseline','Factor1 Fault','Factor2 Baseline','Factor2 Fault'],
    y=[fa_var_base[0], fa_var_fault[0], fa_var_base[1], fa_var_fault[1]],
    marker_color=['steelblue','tomato','steelblue','tomato'],
    name='FA Factor Variance'
), row=2, col=2)

fig.update_layout(
    height=750,
    width=1200,
    template='plotly_white',
    title='Q4: PCA vs FA Robustness to Single Sensor Fault'
)

fig.show()

QUESTION 4 ANALYSIS

--- Sensor Uniqueness (φ²) Noise Floor Metrics ---
Sensor_1: Uniqueness φ²=0.50%  |  Communality=99.50%
Sensor_2: Uniqueness φ²=24.03%  |  Communality=75.97%
Sensor_3: Uniqueness φ²=41.65%  |  Communality=58.35%
Sensor_4: Uniqueness φ²=99.42%  |  Communality=0.58%

--- PCA Strategy Baseline vs Sensor_4 Fault ---
Baseline  : Mean T²=1.9992  |  Mean Q(SPE)=1.0186
S4 Fault  : Mean T²=10.0369  |  Mean Q(SPE)=1.8061
T² change : 8.0377
Q  change : 0.7875

--- FA Strategy Baseline vs Sensor_4 Fault ---
Factor1 variance: Baseline=0.4053  |  Fault=0.4055  |  Change=0.0001
Factor2 variance: Baseline=0.2446  |  Fault=0.2644  |  Change=0.0198

--- Answer: Which strategy is more robust to a single sensor fault? ---

WINNER: FA Strategy (monitoring rotated latent factor scores)

Justification using Uniqueness φ² metrics:

Sensor_4 has φ² = 99.42% — meaning ~99% of its
variance is idiosyncratic noise completely unrelated to the
latent factors F1 and F2.

PCA Strategy vulnerabilit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

